## Generate Samples Using Sobol Sequence

Notes on ResStock Priors:

Heating setpoint: Gamma distribution fit

Gap: Truncated normal Distribution with mean informed by data, std uses sd fraction

People: Gamma Distribution fit

Infiltration rate: Gamma Distribution fit (use same data for attic, garage, living)

Equipment power: Normal guess, sd fraction

Lighting: mixed normal based on data, uses sd fraction

Heating COP: Normal guess, sd fraction

Fan efficiency: Normal guess, sd fraction

Pressure rise: Normal guess, sd fraction

Solar Transmittance: Gamma Distribution fit

Burner: Normal guess, sd fraction

Vent flow: Normal guess, sd fraction

In [18]:
from pathlib import Path
import time
from eppy.modeleditor import IDF
import pandas as pd
import numpy as np
import shutil
import os
from mpi4py import MPI
from scipy import stats
from scipy.stats import gaussian_kde, gamma, norm, truncnorm
from scipy.interpolate import interp1d
import SALib
from SALib.sample import sobol


In [19]:
# library of functions
import importlib
import resstock_library
importlib.reload(resstock_library)
from resstock_library import *

In [21]:
def generate_sobol_sequence(num_files, seed):
    #  standard deviation for each paramter is 5% of its original value
    sd_frac = 0.05

    # manually create a parameter list
    parameter_names = ['heating_setpoint', 'cooling_setpoint', 'people_per_area', 'infil_flow_rate_living', 'infil_flow_rate_garage', 'infil_flow_rate_attic', 'watts_equip', 'watts_lights', 'heating_COP', 'fan_efficiency', 'pressure_rise', 'solar_transmittance', 'burner_eff', 'vent_flow_rate', 'gap']

    ### dealing with parameters that do not have data informed distributions ### 

    # means for params without distributions 
    means = {
        'watts_equip': 500,
        'heating_COP': 4.0,
        'fan_efficiency': 0.7,
        'pressure_rise': 400.0,
        'burner_eff': 0.8,
        'vent_flow_rate': 0.131944
    }

    # param restrictions
    min_gap = 0 
    max_solar_transmittance = 1-0.075 # max solar transmittance

    # creating a dictionary to store the std
    parameter_std = {}

    # adding std into dictionary
    for name, mean in means.items():
        parameter_std[name] = mean*sd_frac

    parameter_std['burner_eff'] = sd_frac # manually change burner efficiency

    # creating problem dictionary for sobol sampling
    # requires first initially excluding the cooling setpoint because it is dependent on the heating setpoint
    names_for_problem = [n for n in parameter_names if n != 'cooling_setpoint']

    # use 0 to 1 uniform distribution for all bounds
    bounds = []
    for name in names_for_problem:
        bounds.append([0, 1])

    # creating distributions
    dists = ['unif'] * len(names_for_problem)

    # dictionary formatted for Sobol sequence
    problem = {
        'num_vars': len(names_for_problem),
        'names': names_for_problem,
        'bounds': bounds,
        'dists': dists
    }

    # conduct sobol sequence sampling
    N = num_files # baseline number of samples
    param_values = sobol.sample(problem, N, calc_second_order=False, seed = seed) # array with dimensions [N*(P+2), P]

    ### Inverse CDF Transformations ### 

    # extracting uniform samples
    param_idx = {} # dictionary storing indexes via param names as key (used for reinserting cooling setpoint later)
    uniform_samples = {}
    for name in problem['names']:
        param_idx[name] = problem['names'].index(name)
        uniform_samples[name] = param_values[:, param_idx[name]]


    # create dictionary for transformed samples 
    transformed_samples = {}

    # Gamma distribution params
    transformed_samples['heating_setpoint'] = gamma_inverse_cdf(
        uniform_samples['heating_setpoint'],
        seed=seed,
        probs=extract_heating_probs()
    )
    transformed_samples['people_per_area'] = gamma_inverse_cdf(
        uniform_samples['people_per_area'],
        seed=seed,
        probs=extract_people_prior()
    )

    for infil_param in ['infil_flow_rate_garage', 'infil_flow_rate_attic', 'infil_flow_rate_living']:
        transformed_samples[infil_param] = gamma_inverse_cdf(
            uniform_samples[infil_param],
            seed=seed,
            probs=extract_infil_prior()
        )

    transformed_samples['solar_transmittance'] = gamma_inverse_cdf(
        uniform_samples['solar_transmittance'],
        seed=seed,
        probs = extract_solar_probs()[["solar_transmittance", "probability"]]
    )
    # enforce max solar transmittance (set all values above max to the max)
    transformed_samples['solar_transmittance'] = transformed_samples[
        'solar_transmittance'
    ].clip(max=max_solar_transmittance)

    # Normal Distribution params 
    for param in ['watts_equip','heating_COP', 'fan_efficiency', 'pressure_rise', 'burner_eff', 'vent_flow_rate']:
        transformed_samples[param] = normal_inverse_cdf(
            uniform_samples[param],
            means=means,
            parameter_std=parameter_std,
            param=param
        )

    # Truncated Normal Distribution
    transformed_samples['gap'] = truncated_normal_inverse_cdf(
        uniform_samples['gap'],
        extract_gap_info(seed=seed)[0], # mean
        extract_gap_info(seed=seed)[1], # std
        lower_bound=min_gap,
        upper_bound=np.inf
    )

    # Mixed noraml Distribution
    transformed_samples['watts_lights'] = mixed_normal_inverse_cdf(
        uniform_samples['watts_lights'],
        probs = extract_lighting_probs()
    )

    ### adding cooling setpoint using gap data
    transformed_samples['cooling_setpoint'] = transformed_samples['heating_setpoint'] + transformed_samples['gap']

    ### creating new array with transformed samples
    transformed_param_values = np.column_stack(
        [transformed_samples[param] for param in parameter_names]
    )
    transformed_param_values = np.delete(transformed_param_values, param_idx['gap']+1, axis=1) # deleting the gap column of values
    parameter_names.remove('gap') # delete gap from list of parameters

    ### Creating samples
    # convert param_values into a list of dictionaries, where the keys correspond to the input parameters
    samples = []
    for i in range(len(transformed_param_values)):
        sample_dict = {}
        for j in range(param_values.shape[1]):
            sample_dict[parameter_names[j]] = transformed_param_values[i,j]
        samples.append(sample_dict)

    ### validating samples 
    invalid_samples = []
    for i, sdict in enumerate(samples):
        # making sure heating point is below cooling point
        if  (sdict['cooling_setpoint'] - sdict['heating_setpoint']) < min_gap:
            print('Setpoint Error')
            invalid_samples.append(i)

    if not invalid_samples:
        print("No invalid samples")
    else:
        print("Invalid samples:", invalid_samples)

    ### returning outputs
    return samples

In [22]:
generate_sobol_sequence(num_files = 1000, seed = 1)

/jumbo/keller-lab/Applications/mambaforge/envs/eplus/lib/python3.9/site-packages/scipy/stats/_qmc.py:804: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  sample = self._random(n, workers=workers)


No invalid samples


[{'heating_setpoint': 19.304160728877967,
  'cooling_setpoint': 20.66423859147513,
  'people_per_area': 2.5867127781630272,
  'infil_flow_rate_living': 0.1229456242703889,
  'infil_flow_rate_garage': 0.06907758353982678,
  'infil_flow_rate_attic': 0.17938206158697237,
  'watts_equip': 484.11322095372964,
  'watts_lights': 724.9303769727129,
  'heating_COP': 3.93714885591078,
  'fan_efficiency': 0.721601765183844,
  'pressure_rise': 399.5479517023524,
  'solar_transmittance': 0.6536996984294464,
  'burner_eff': 0.7925204629529131,
  'vent_flow_rate': 0.12643374656566791},
 {'heating_setpoint': 20.639888516596667,
  'cooling_setpoint': 21.99996637919383,
  'people_per_area': 2.5867127781630272,
  'infil_flow_rate_living': 0.1229456242703889,
  'infil_flow_rate_garage': 0.06907758353982678,
  'infil_flow_rate_attic': 0.17938206158697237,
  'watts_equip': 484.11322095372964,
  'watts_lights': 724.9303769727129,
  'heating_COP': 3.93714885591078,
  'fan_efficiency': 0.721601765183844,
  'pr